# Shrinkage detector, round 4: a test that can actually fail

Round 3's Test B could not reveal much. On that simulator a method with **no adjustment at all** is within 1% of the truth at effect 128, and CausalPFN rescales each treatment group separately, so with no useful signal it returns that same unadjusted answer rather than zero.

This notebook replaces it with:

| Test | Question | Needs GPU? |
|---|---|---|
| **M. Metamorphic audit** | Which simple data changes does CausalPFN follow exactly, and which tell us anything? | Yes |
| **C. Anchor experiment** | On simulators where skipping adjustment gives a wrong answer, which way is CausalPFN off? | Yes |

It also compares three detectors: the round 3 z-test, the same test with a 10% margin, and a rectifier test whose variance accounts for both estimates using the same data.

**Speed.** Datasets of the same size share one GPU pass in half precision, and CausalPFN's CPU helper model is skipped when it has no effect (when no treatment group exceeds 1,024 units). Section 4 checks this against the normal code before anything runs. AIPW can only use the CPU, so its results are stored in a cache: if `round4_aipw.csv` and `round4_propensities.npz` are uploaded to `jackfruit_sim`, it is skipped, and otherwise it runs here.

**Before running:** `Runtime > Change runtime type > T4 GPU`, then `Runtime > Run all`. Runs can be resumed: finished datasets are kept in the CSV as long as the runtime survives.

## 1. Setup

In [ ]:
!pip install -q causalpfn==0.1.5

import os, itertools, time, warnings
warnings.filterwarnings('ignore')
OUT_DIR = './jackfruit_sim'   # Colab's local folder (cleared if the runtime disconnects)
os.makedirs(OUT_DIR, exist_ok=True)
print('Saving results to', OUT_DIR)

## 2. Config

In [ ]:
RUN_M = True
RUN_C = True
RUN_NAME = 'round4'

# Test M: metamorphic audit
M_DGPS   = ['notebook', 'tau_on_confounder']
M_NS     = [500, 2000]
M_SEEDS  = range(0, 10)
M_EFFECT = 4.0
M_DELTAS = [1.0, 8.0, 64.0]             # constant planted effects
M_KS     = [0.5, 1.0, 2.0, 4.0]         # size of the covariate-dependent planted effect, in SDs of Y

# Test C: anchor experiment. Seeds are the outer loop, so a partial run is still balanced.
C_DGPS    = ['notebook', 'tau_on_confounder', 'flipped', 'scaled']
C_EFFECTS = [1.0, 2.0, 4.0, 8.0, 32.0, 128.0]
C_NS      = [500, 2000]
C_SEEDS   = range(0, 30)
C_K       = 1.0                         # planted-effect size used in Test C

GPU_TOKENS = 200_000                    # rows per GPU pass (context + 2 x query); about 3.5 GB on a T4
MAX_ARM    = 1024                       # above this CausalPFN must pick context with its CPU helper model

M_CSV = f'{OUT_DIR}/{RUN_NAME}_M_metamorphic.csv'
C_CSV = f'{OUT_DIR}/{RUN_NAME}_C_anchor.csv'
AIPW_CSV = f'{OUT_DIR}/{RUN_NAME}_aipw.csv'
PROP_NPZ = f'{OUT_DIR}/{RUN_NAME}_propensities.npz'
print('Test M datasets:', len(M_DGPS)*len(M_NS)*len(M_SEEDS))
print('Test C datasets:', len(C_DGPS)*len(C_EFFECTS)*len(C_NS)*len(C_SEEDS))

## 3. Core code
Same simulator, AIPW and answer key as rounds 1-3, with these additions: `dgp` picks the simulator, the margin and rectifier detectors, the planted-effect self-test and the AIPW cache. The round 3 `detector` is unchanged so the comparison is fair.

In [ ]:
import os
import numpy as np
import pandas as pd
from scipy.special import expit
from sklearn.model_selection import KFold
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.ensemble import HistGradientBoostingRegressor, HistGradientBoostingClassifier

D = 6  # number of covariates
DGPS = ["notebook", "tau_on_confounder", "flipped", "scaled"]

# ---------- 1. Data generator (we know the truth) ----------
def simulate(n, effect_size, overlap, seed, noise_sd=1.0, dgp="notebook"):
    """effect_size = true ATE. overlap: 'good' or 'poor'. noise_sd = outcome noise.
    dgp picks the design:
      notebook:          same as rounds 1-3. Confounding stays fixed while the effect grows.
      tau_on_confounder: the effect size depends on X0, which also drives treatment.
      flipped:           confounding runs the other way (the naive contrast is too low).
      scaled:            confounding grows with the effect, so it never becomes negligible.
    The random draws match rounds 1-3 for the same seed.
    """
    rng = np.random.default_rng(seed)
    X = rng.normal(0, 1, size=(n, D))
    # Nonlinear confounding: the same features drive treatment and outcome
    score = X[:, 0] - 0.5 * X[:, 1] + 0.8 * (X[:, 2] ** 2 - 1) + 0.5 * X[:, 0] * X[:, 3]
    if dgp == "flipped":
        score = -score
    strength = 0.6 if overlap == "good" else 2.0   # bigger = more extreme propensities
    e = expit(strength * score)
    T = rng.binomial(1, e)
    mu0 = 1.0 * X[:, 0] + np.sin(2 * X[:, 1]) + 0.7 * X[:, 2] ** 2 + 0.5 * X[:, 3] * X[:, 4]
    if dgp == "scaled":
        mu0 = mu0 - effect_size * X[:, 0]
    if dgp == "tau_on_confounder":
        tau = effect_size * (1 - 0.5 * X[:, 0])
    else:
        tau = effect_size * (1 + 0.5 * X[:, 4])        # heterogeneous effect, mean = effect_size
    Y = mu0 + T * tau + rng.normal(0, noise_sd, size=n)
    true_ate = float(tau.mean())                       # sample ATE (the "answer key")
    return X, T, Y, e, true_ate, mu0, mu0 + tau

# ---------- 2. AIPW with cross-fitting ----------
def _outcome_model(kind, seed):
    if kind == "flexible":
        return HistGradientBoostingRegressor(max_iter=200, learning_rate=0.05, random_state=seed)
    return LinearRegression()          # 'wrong': linear model on a nonlinear truth

def _prop_model(kind, seed):
    if kind == "flexible":
        return HistGradientBoostingClassifier(max_iter=200, learning_rate=0.05, random_state=seed)
    return LogisticRegression(max_iter=1000)   # 'wrong': misses the squared / interaction terms

NUISANCE_SPECS = {
    "both_ok":       ("flexible", "flexible"),
    "outcome_wrong": ("wrong",    "flexible"),
    "prop_wrong":    ("flexible", "wrong"),
    "both_wrong":    ("wrong",    "wrong"),
}

def crossfit_propensity(X, T, seed, folds=5, kind="flexible"):
    e_hat = np.zeros(len(T))
    for tr, te in KFold(folds, shuffle=True, random_state=seed).split(X):
        e_hat[te] = _prop_model(kind, seed).fit(X[tr], T[tr]).predict_proba(X[te])[:, 1]
    return e_hat

def aipw(X, T, Y, spec, seed, folds=5, clip=0.01, e_hat=None):
    """Returns ATE estimate, its standard error, fitted propensities and held-out outcome predictions.
    Pass e_hat to reuse propensities that were already fitted on the same X, T, seed and folds."""
    out_kind, prop_kind = NUISANCE_SPECS[spec]
    n = len(Y)
    if e_hat is None:
        e_hat = crossfit_propensity(X, T, seed, folds, prop_kind)
    mu0_hat, mu1_hat = np.zeros(n), np.zeros(n)
    for tr, te in KFold(folds, shuffle=True, random_state=seed).split(X):
        for t, store in ((0, mu0_hat), (1, mu1_hat)):
            idx = tr[T[tr] == t]
            om = _outcome_model(out_kind, seed).fit(X[idx], Y[idx])
            store[te] = om.predict(X[te])
    e_c = np.clip(e_hat, clip, 1 - clip)
    psi = (mu1_hat - mu0_hat
           + T * (Y - mu1_hat) / e_c
           - (1 - T) * (Y - mu0_hat) / (1 - e_c))       # influence-function scores
    ate = psi.mean()
    se = psi.std(ddof=1) / np.sqrt(n)
    return float(ate), float(se), e_hat, mu0_hat, mu1_hat

# ---------- 3. The detectors ----------
def extreme_share(e_hat):
    return float(np.mean((e_hat < 0.02) | (e_hat > 0.98)))

def detector(cfm_ate, aipw_ate, aipw_se, e_hat, alpha_z=1.645, overlap_tol=0.05):
    """Rounds 1-3: flags any gap toward zero bigger than AIPW's noise. Unchanged, kept as the baseline."""
    if extreme_share(e_hat) > overlap_tol:
        return "uncertain", np.nan
    gap = aipw_ate - cfm_ate
    z = np.sign(aipw_ate) * gap / max(aipw_se, 1e-12)   # >0 means CFM sits closer to 0
    return ("flag" if z > alpha_z else "no_flag"), float(z)

def _verdict(z, alpha_z):
    if z > alpha_z:
        return "shrunk"
    return "not_shrunk" if z < -alpha_z else "inconclusive"

def margin_detector(cfm_ate, aipw_ate, aipw_se, e_hat, margin=0.10, alpha_z=1.645, overlap_tol=0.05):
    """Tests the claim the label makes: is the CFM more than `margin` closer to zero than the truth?
    Fixes the null, but still uses AIPW's SE alone."""
    if extreme_share(e_hat) > overlap_tol:
        return "uncertain", np.nan
    z = np.sign(aipw_ate) * ((1 - margin) * aipw_ate - cfm_ate) / max((1 - margin) * aipw_se, 1e-12)
    return _verdict(z, alpha_z), float(z)

def rectifier_scores(mu0_hat, mu1_hat, T, Y, e_hat, clip=0.01):
    """Each unit's correction to the model's own held-out predictions, weighted as in AIPW."""
    e_c = np.clip(e_hat, clip, 1 - clip)
    return T * (Y - mu1_hat) / e_c - (1 - T) * (Y - mu0_hat) / (1 - e_c)

def rectifier_detector(mu0_hat, mu1_hat, T, Y, e_hat, margin=0.10, alpha_z=1.645, overlap_tol=0.05):
    """Margin test on a sample mean. mu0_hat and mu1_hat must be held-out predictions.
    The mean of the scores is (1-margin)*ATE - CFM ATE, and their spread is its own SE,
    so the variance accounts for both estimates using the same data."""
    if extreme_share(e_hat) > overlap_tol:
        return "uncertain", np.nan, np.nan
    r = rectifier_scores(mu0_hat, mu1_hat, T, Y, e_hat)
    cate = mu1_hat - mu0_hat
    psi = (1 - margin) * r - margin * cate
    n = len(Y)
    s = np.sign(cate.mean() + r.mean())
    z = s * psi.mean() / max(psi.std(ddof=1) / np.sqrt(n), 1e-12)
    z0 = s * r.mean() / max(r.std(ddof=1) / np.sqrt(n), 1e-12)       # no margin: >0 toward zero, <0 away from it
    return _verdict(z, alpha_z), float(z), float(z0)

# ---------- 4. Ground-truth labels ----------
def truly_shrunk(cfm_ate, true_ate, rel_tol=0.10):
    """CFM estimate is pulled toward zero by more than 10% of the true effect."""
    return bool(np.sign(true_ate) * (true_ate - cfm_ate) > rel_tol * abs(true_ate))

def truly_inflated(cfm_ate, true_ate, rel_tol=0.10):
    """CFM estimate is pushed away from zero by more than 10% of the true effect."""
    return bool(np.sign(true_ate) * (cfm_ate - true_ate) > rel_tol * abs(true_ate))

# ---------- 5. Planted-effect self-test ----------
def planted_effect(X, Y, k):
    """A known effect that varies with X0, which drives treatment, centred so the true ATE does not move.
    Built from X rather than estimated propensities: those also set the AIPW weights, which made the
    plant largest exactly where the weights were most extreme. Y + T*delta needs no reference to judge."""
    x = X[:, 0]
    return k * Y.std() * (x - x.mean()) / x.std()

def naive_contrast(T, Y):
    return float(Y[T == 1].mean() - Y[T == 0].mean())

# ---------- 6. Results helpers ----------
def done_keys(path, cols):
    if not os.path.exists(path):
        return set()
    prev = pd.read_csv(path)
    return set(map(tuple, prev[cols].drop_duplicates().values.tolist()))

# ---------- 7. AIPW cache: all the CPU-only work, done once ----------
def treatment_key(seed, n, dgp):
    """Treatment depends only on X and the direction of confounding, not on the effect size."""
    return f"{seed}_{n}_{int(dgp == 'flipped')}"

def _propensity_job(key, dgp, n, seed):
    X, T, *_ = simulate(n, 1.0, "good", seed, dgp=dgp)
    return key, crossfit_propensity(X, T, seed)

def _aipw_job(seed, dgp, es, n, e_hat):
    X, T, Y, *_ = simulate(n, es, "good", seed, dgp=dgp)
    a, se, *_ = aipw(X, T, Y, "both_ok", seed, e_hat=e_hat)
    return dict(dgp=dgp, effect_size=es, n=n, seed=seed, aipw_ate=a, aipw_se=se, y_check=float(Y.sum()))

def build_aipw_cache(grid, csv_path, npz_path, n_jobs=-1):
    """grid holds (seed, dgp, effect_size, n). Finished work is kept, so this can be re-run or run elsewhere first.
    Propensity models are fitted once per (seed, n, confounding direction) instead of once per dataset,
    and each worker uses one thread because boosting scales better across datasets than within one fit."""
    from joblib import Parallel, delayed, parallel_config
    props = dict(np.load(npz_path)) if os.path.exists(npz_path) else {}
    todo = {treatment_key(s, n, d): (d, n, s) for s, d, es, n in grid if treatment_key(s, n, d) not in props}
    done = done_keys(csv_path, ["seed", "dgp", "effect_size", "n"])
    rest = [g for g in grid if g not in done]
    with parallel_config(backend="loky", inner_max_num_threads=1):
        if todo:
            props.update(Parallel(n_jobs=n_jobs)(delayed(_propensity_job)(k, *v) for k, v in todo.items()))
            np.savez(npz_path, **props)
        for i in range(0, len(rest), 200):       # saved in blocks so an interruption loses little
            rows = Parallel(n_jobs=n_jobs)(delayed(_aipw_job)(s, d, es, n, props[treatment_key(s, n, d)])
                                           for s, d, es, n in rest[i:i + 200])
            pd.DataFrame(rows).to_csv(csv_path, mode="a", header=not os.path.exists(csv_path), index=False)
    return props

## 4. CausalPFN on the GPU
`causalpfn` is pinned to 0.1.5 because this code depends on its internals. `cfm_predict` runs many datasets through one GPU pass, and uses the normal route only when a treatment group is too big to fit in the context. Cross-fitted predictions come from a context that excludes the unit being predicted. Without this, a model can see a unit's own outcome, and the rectifier test would not be valid.

In [ ]:
import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)
if (RUN_M or RUN_C) and device.type != 'cuda':
    print('WARNING: no GPU. Switch runtime to T4 GPU, otherwise this will be very slow.')

from causalpfn import ATEEstimator
from sklearn.base import clone

class SharedWeightsATE(ATEEstimator):
    """Fresh estimator per dataset, but the network weights are loaded onto the GPU only once.
    (fit() normally re-reads the checkpoint from disk every call. The round-2 bug came from
    estimate_ate() caching mu_0/mu_1, which fit() never resets, so a fresh object is still needed.)"""
    _cache = None
    def load_model(self):
        if SharedWeightsATE._cache is None:
            super().load_model()
            SharedWeightsATE._cache = (self.icl_model, self.max_feature_size, self.x_dim_transformer)
        self.icl_model, self.max_feature_size, xdt = SharedWeightsATE._cache
        self.x_dim_transformer = clone(xdt)

def cfm_fit(X, T, Y):
    # A FRESH estimator every call. Reusing one object returned the first dataset's answer forever (round 2 bug).
    est = SharedWeightsATE(device=device, verbose=False)
    est.fit(X.astype(np.float32), T.astype(np.float32), Y.astype(np.float32))
    return est

def cfm_ate(X, T, Y):
    return float(cfm_fit(X, T, Y).estimate_ate())

def _to_gpu(arrays):
    return torch.from_numpy(np.stack(arrays).astype(np.float32)).to(device, non_blocking=True)

def _query_treatment(m):
    return np.concatenate([np.zeros(m, dtype=np.float32), np.ones(m, dtype=np.float32)])

def _predict_one(Xc, Tc, Yc, Xq, amp):
    # The normal route: trains the CPU helper model that picks which context each query sees.
    est = cfm_fit(Xc, Tc, Yc)
    Xq = Xq.astype(np.float32)
    with torch.autocast('cuda', dtype=torch.float16, enabled=amp and device.type == 'cuda'):
        pred = est._predict_cepo(X_context=est.X_train, t_context=est.t_train, y_context=est.y_train,
                                 X_query=np.concatenate([Xq, Xq]), t_query=_query_treatment(len(Xq)),
                                 temperature=est.prediction_temperature)
    m = len(Xq)
    return np.asarray(pred[:m], dtype=np.float64), np.asarray(pred[m:], dtype=np.float64)

@torch.inference_mode()
def _predict_batch(jobs, amp):
    # When every unit fits in the context the helper model changes nothing, so skip it and batch.
    Xc, Tc, Yc, Xq, Tq = [], [], [], [], []
    for X_ctx, T_ctx, Y_ctx, X_q in jobs:
        order = np.concatenate([np.where(T_ctx == 0)[0], np.where(T_ctx == 1)[0]])   # same order as the normal route
        Xc.append(X_ctx[order]); Tc.append(T_ctx[order]); Yc.append(Y_ctx[order])
        Xq.append(np.concatenate([X_q, X_q])); Tq.append(_query_treatment(len(X_q)))
    with torch.autocast('cuda', dtype=torch.float16, enabled=amp and device.type == 'cuda'):
        pred = ICL.predict_cepo(X_context=_to_gpu(Xc), t_context=_to_gpu(Tc), y_context=_to_gpu(Yc),
                                X_query=_to_gpu(Xq), t_query=_to_gpu(Tq), temperature=TEMPERATURE)
    pred = pred[:, 0, :].float().cpu().numpy().astype(np.float64)
    m = pred.shape[1] // 2
    return [(p[:m], p[m:]) for p in pred]

def cfm_predict(jobs, amp=True):
    """jobs holds (X_context, T_context, Y_context, X_query). Returns (mu0, mu1) for each job's query."""
    out, groups = [None] * len(jobs), {}
    for i, (_, Tc, _, Xq) in enumerate(jobs):
        if max(Tc.sum(), len(Tc) - Tc.sum()) <= MAX_ARM:
            groups.setdefault((len(Tc), len(Xq)), []).append(i)
        else:
            out[i] = _predict_one(*jobs[i], amp)
    for (ctx, q), idx in groups.items():
        size = max(1, GPU_TOKENS // (ctx + 2 * q))
        for s in range(0, len(idx), size):
            part = idx[s:s + size]
            for i, r in zip(part, _predict_batch([jobs[i] for i in part], amp)):
                out[i] = r
    return out

def ate_s(mu0, mu1):
    return float((mu1 - mu0).mean())

def ate_ra(mu0, mu1, T, Y):
    return float((np.where(T == 1, Y, mu1) - np.where(T == 0, Y, mu0)).mean())

# Sanity check: two very different effects must give very different estimates
checks = []
for es in [4.0, 32.0]:
    X, T, Y, e, ta, _, _ = simulate(1000, es, 'good', 123)
    t0 = time.time()
    c = cfm_ate(X, T, Y)
    checks.append(c)
    print(f'sanity check: true ATE = {ta:.2f}, CausalPFN ATE = {c:.2f}  ({time.time()-t0:.1f}s)')
assert abs(checks[1] - checks[0]) > 10, 'CausalPFN gave nearly the same answer for effect 4 and 32. Something is cached; stop and report.'
print('Sanity check passed: estimates change with the data.')

ICL = SharedWeightsATE._cache[0].train(False)
TEMPERATURE = torch.tensor([1.0], device=device)

# Speed check: the batched route must match the normal route
for n, es, dgp in [(500, 4.0, 'notebook'), (500, 128.0, 'scaled'), (2000, 8.0, 'flipped')]:
    data = [simulate(n, es, 'good', s, dgp=dgp) for s in range(0, 40)]
    data = [d for d in data if max(d[1].sum(), n - d[1].sum()) <= MAX_ARM][:3]
    fast = cfm_predict([(d[0], d[1], d[2], d[0]) for d in data], amp=False)
    for d, (m0, m1) in zip(data, fast):
        gap = abs(ate_s(m0, m1) - cfm_ate(d[0], d[1], d[2]))
        assert gap < 1e-3 * d[2].std(), f'Batched route differs from the normal route by {gap}. Stop and report.'
print('Speed check passed: the batched route matches the normal route.')

grid_c = list(itertools.product(C_SEEDS, C_DGPS, C_EFFECTS, C_NS))
t0 = time.time()
props = build_aipw_cache(grid_c, AIPW_CSV, PROP_NPZ, n_jobs=os.cpu_count())   # returns at once if already uploaded
aipw_tbl = pd.read_csv(AIPW_CSV).drop_duplicates(['seed', 'dgp', 'effect_size', 'n']).set_index(['seed', 'dgp', 'effect_size', 'n']).sort_index()
print(f'AIPW cache ready: {len(aipw_tbl)} datasets ({time.time()-t0:.0f}s)')

## 5. Test M: metamorphic audit
Change the data in a way whose effect on the right answer is known, then check whether CausalPFN follows. Scaling Y, shifting Y and adding a constant effect should follow exactly, because CausalPFN rescales each group. If they do, those checks say nothing about adjustment. The covariate-dependent plant has a true ATE change of zero, while a model that does no adjustment would move by `naive_shift`. Full precision here, because these are checks of exact equality.

In [ ]:
if RUN_M:
    done = done_keys(M_CSV, ['dgp', 'n', 'seed'])
    grid = [g for g in itertools.product(M_SEEDS, M_DGPS, M_NS) if (g[1], g[2], g[0]) not in done]
    t0 = time.time()
    for i, (seed, dgp, n) in enumerate(grid):
        X, T, Y, e, ta, _, _ = simulate(n, M_EFFECT, 'good', seed, dgp=dgp)
        eh = props[treatment_key(seed, n, dgp)]
        deltas = {k: planted_effect(X, Y, k) for k in M_KS}
        ys = {'cfm_base': Y, 'cfm_y_times_10': Y * 10, 'cfm_y_plus_100': Y + 100,
              **{f'cfm_plus_{d:g}T': Y + d * T for d in M_DELTAS},
              **{f'cfm_plant_{k:g}': Y + T * deltas[k] for k in M_KS}}
        preds = cfm_predict([(X, T, y, X) for y in ys.values()], amp=False)
        row = dict(dgp=dgp, n=n, seed=seed, true_ate=ta, naive=naive_contrast(T, Y), max_arm=int(max(T.sum(), n - T.sum())),
                   **{name: ate_s(*p) for name, p in zip(ys, preds)},
                   **{f'naive_shift_{k:g}': float(deltas[k][T == 1].mean()) for k in M_KS})
        pd.DataFrame([row]).to_csv(M_CSV, mode='a', header=not os.path.exists(M_CSV), index=False)
    print(f'Test M: {len(grid)} datasets in {time.time()-t0:.0f}s', flush=True)

dfM = pd.read_csv(M_CSV)
dev = pd.DataFrame({
    'Y*10 (should be 10x)':  (dfM.cfm_y_times_10 - 10 * dfM.cfm_base).abs(),
    'Y+100 (should not move)': (dfM.cfm_y_plus_100 - dfM.cfm_base).abs(),
    **{f'Y+{d:g}T (should add {d:g})': (dfM[f'cfm_plus_{d:g}T'] - dfM.cfm_base - d).abs() for d in M_DELTAS},
})
print('Largest absolute deviation from the exact answer, by n and whether a group exceeds 1,024 units:')
display(dev.groupby([dfM.n, dfM.max_arm > MAX_ARM]).max().round(4))

## 6. Test C: anchor experiment
**Anchor fraction** = `(CausalPFN - naive) / (truth - naive)`. 1 means full adjustment and 0 means CausalPFN returned the naive contrast. It is only meaningful where the naive contrast is more than 10% wrong.

In [ ]:
if RUN_C:
    done = done_keys(C_CSV, ['dgp', 'effect_size', 'n', 'seed'])
    t0 = time.time()
    finished = 0
    for seed, n in itertools.product(C_SEEDS, C_NS):       # one seed and size at a time, all of it batched
        todo = [(d, es) for d, es in itertools.product(C_DGPS, C_EFFECTS) if (d, es, n, seed) not in done]
        if not todo:
            continue
        data, jobs = [], []
        folds = list(KFold(5, shuffle=True, random_state=seed).split(np.zeros((n, 1))))
        for dgp, es in todo:
            X, T, Y, e, ta, _, _ = simulate(n, es, 'good', seed, dgp=dgp)
            eh = props[treatment_key(seed, n, dgp)]
            delta = planted_effect(X, Y, C_K)
            data.append((dgp, es, X, T, Y, ta, eh, delta))
            jobs.append((X, T, Y, X))
            jobs.append((X, T, Y + T * delta, X))
            jobs.extend((X[tr], T[tr], Y[tr], X[te]) for tr, te in folds)
        preds = cfm_predict(jobs, amp=True)
        rows = []
        for i, (dgp, es, X, T, Y, ta, eh, delta) in enumerate(data):
            base, plant, *cf = preds[i * 7:(i + 1) * 7]
            mu0_cf, mu1_cf = np.zeros(n), np.zeros(n)
            for (_, te), (m0, m1) in zip(folds, cf):
                mu0_cf[te], mu1_cf[te] = m0, m1
            c, c_cf = ate_s(*base), ate_s(mu0_cf, mu1_cf)
            r = aipw_tbl.loc[(seed, dgp, es, n)]
            assert abs(r.y_check - Y.sum()) < 1e-6 * max(abs(Y.sum()), 1), 'AIPW cache was built on different data. Stop and report.'
            a, se = float(r.aipw_ate), float(r.aipw_se)
            d0, z0 = detector(c, a, se, eh)
            dm, zm = margin_detector(c, a, se, eh)
            dr, zr, zr0 = rectifier_detector(mu0_cf, mu1_cf, T, Y, eh)
            rows.append(dict(dgp=dgp, effect_size=es, n=n, seed=seed, true_ate=ta, naive=naive_contrast(T, Y),
                             cfm_ate=c, cfm_ate_ra=ate_ra(*base, T, Y), cfm_ate_cf=c_cf, aipw_ate=a, aipw_se=se,
                             y_sd=float(Y.std()), extreme_share=extreme_share(eh), max_arm=int(max(T.sum(), n - T.sum())),
                             decision_old=d0, z_old=z0, decision_margin=dm, z_margin=zm,
                             decision_rect=dr, z_rect=zr, z_rect_nomargin=zr0,
                             truly_shrunk=truly_shrunk(c, ta), truly_inflated=truly_inflated(c, ta),
                             truly_shrunk_cf=truly_shrunk(c_cf, ta), truly_inflated_cf=truly_inflated(c_cf, ta),
                             cfm_plant=ate_s(*plant), naive_shift=float(delta[T == 1].mean())))
        pd.DataFrame(rows).to_csv(C_CSV, mode='a', header=not os.path.exists(C_CSV), index=False)
        finished += len(rows)
        print(f'Test C seed {seed} n={n}: {finished} datasets, {time.time()-t0:.0f}s', flush=True)

dfC = pd.read_csv(C_CSV)
dfC['rel_bias'] = (dfC.cfm_ate - dfC.true_ate) / dfC.true_ate.abs()
dfC['naive_rel_bias'] = (dfC.naive - dfC.true_ate) / dfC.true_ate.abs()
print('Seeds finished per setting:', dfC.groupby(['dgp', 'effect_size', 'n']).size().min(), 'to', dfC.groupby(['dgp', 'effect_size', 'n']).size().max())
print('Mean CausalPFN relative bias (signed):')
display(dfC.pivot_table(index=['dgp', 'n'], columns='effect_size', values='rel_bias', aggfunc='mean').round(3))
print('Mean naive relative bias (signed), for comparison:')
display(dfC.pivot_table(index=['dgp', 'n'], columns='effect_size', values='naive_rel_bias', aggfunc='mean').round(3))
open(f'{OUT_DIR}/{RUN_NAME}_done.txt', 'w').write('done')